# Regularization

If we face overfitting we are supposed to regularize models. There are serveral ways to do this.

## Preparing the data (Fashion MNIST)

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf 
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import InputLayer
print("Tensorflow version:", tf.__version__)

#read data with pandas
dataTrain = pd.read_csv('data/fashion-mnist_train.csv')
dataTest = pd.read_csv('data/fashion-mnist_test.csv')
#convert to numpy and initialize variables for training
X_train, y_train = dataTrain.iloc[:, 1:].to_numpy(), dataTrain.iloc[:, 0].to_numpy()
X_test, y_test = dataTest.iloc[:, 1:].to_numpy(), dataTest.iloc[:, 0].to_numpy()

#Define Classnames
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# prepare ground truth as one-hot encoded values
y_one_hot = tf.one_hot(y_train,len(class_names))
y_one_hot_test = tf.one_hot(y_test,len(class_names))

print("Fashion Mnist")
print(f"Training data size: {X_train.shape}")
print(f"Test data size:     {X_test.shape}")
print(f"One Hot encoded:    {y_one_hot.shape}")

# Original Run / Reference Run

In [ ]:
model = Sequential()
model.add(InputLayer((784,)))
model.add(Dense(100, activation="sigmoid"))
model.add(Dense(len(class_names), activation="softmax"))
model.compile(optimizer='sgd', loss="categorical_crossentropy", metrics=["accuracy"])

model.fit(
    X_train,
    y_one_hot,
    epochs=10,
    batch_size=100)

print("\nEvaluation on Test Data")
print(model.evaluate(X_test, y_one_hot_test))

# Now make the model overfitting

Complex Model (more Layers, more neurons) are more likely to overfit. Less data is more likely to overfit. We raise the number of epochs to learn longer. The learning rate has to be adjusted in order to make the model learn effectively.

In [ ]:
from tensorflow.keras.optimizers import SGD
model = Sequential([
      InputLayer((784,)),
      Dense(512, activation='relu'),
      Dense(512, activation='relu'),
      Dense(10, activation="softmax")
      ])
model.compile(optimizer=SGD(learning_rate=0.001), loss="categorical_crossentropy", metrics=['accuracy'])
model.fit(
    X_train[0:10000],
    y_one_hot[0:10000],
    epochs=25,
    batch_size=100)


In [ ]:
print("Evaluation on Training Data")
print(model.evaluate(X_test, y_one_hot_test, batch_size=1000))

# Dropout Layer - Instantiation and Effects

A dropout layer randomly disables a portion of neurons during training so they temporarily do not contribute to the computation. This helps prevent overfitting and usually improves the model’s ability to generalize to unseen data.

In [ ]:
from tensorflow.keras.layers import Dropout 

dropout = Dropout(rate=0.2)
dropout

In [ ]:
# Make the effect visible 
X_train_28x28 = X_train[0:2].reshape(-1,28,28).astype(float)
modified = dropout(X_train_28x28, training=True)   # Class is used as a function, actually this calls dropout.__call__()

fig, axs = plt.subplots(2, 2, figsize=(4, 4))

for index, img in enumerate(X_train_28x28):
    axs[index,0].imshow(img, cmap="gray_r")
    axs[index,1].imshow(modified[index], cmap="gray_r")


# Regularization with Dropout

To use it in a network we just insert it between other layers.

In [ ]:
from tensorflow.keras.optimizers import SGD
model = Sequential([
      InputLayer((784,)),
      Dropout(rate=0.2),
      Dense(512, activation='relu'),
      Dense(512, activation='relu'),
      Dense(10, activation="softmax")
      ])

model.compile(optimizer=SGD(learning_rate=0.001), loss="categorical_crossentropy", metrics=['accuracy'])

model.fit(
    X_train[0:10000],
    y_one_hot[0:10000],
    epochs=60,
    batch_size=100)

In [ ]:
print("Evaluation on Training Data")
print(model.evaluate(X_test, y_one_hot_test, batch_size=1000))

# Regularization with Weight penalties

In [ ]:
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.regularizers import L2 
model = Sequential([
      InputLayer((784,)),
      Dense(512, activation='relu', kernel_regularizer=L2(0.2)),
      Dense(512, activation='relu', kernel_regularizer=L2(0.2)),
      Dense(10, activation="softmax")
      ])

model.compile(optimizer=SGD(learning_rate=0.001), loss="categorical_crossentropy", metrics=['accuracy'])

model.fit(
    X_train[0:10000],
    y_one_hot[0:10000],
    epochs=60,
    batch_size=100)

In [ ]:
print("Evaluation on Training Data")
print(model.evaluate(X_test, y_one_hot_test, batch_size=1000))

# Now both 

In [ ]:
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.regularizers import L2 
model = Sequential([
      InputLayer((784,)),
      Dropout(rate=0.1),
      Dense(512, activation='relu', kernel_regularizer=L2(0.2)),
      Dense(512, activation='relu', kernel_regularizer=L2(0.2)),
      Dropout(rate=0.1),
      Dense(10, activation="softmax")
      ])

model.compile(optimizer=SGD(learning_rate=0.001), loss="categorical_crossentropy", metrics=['accuracy'])

model.fit(
    X_train[0:10000],
    y_one_hot[0:10000],
    epochs=60,
    batch_size=100)

print("Evaluation on Test Data")
print(model.evaluate(X_test, y_one_hot_test, batch_size=1000))